# PyroPredict — Phase 5: INT8 Quantization and Benchmarking

**CMPE 258 Deep Learning** | Spring 2026 | SJSU

This notebook quantizes the best Phase 4 model to INT8 ONNX and benchmarks
FP32 vs INT8 for model size, latency, throughput, and accuracy on the test set.

**Runtime:** CPU is sufficient (GPU optional)  
**Time:** ~20–40 minutes

## 0 — Setup

In [ ]:
!pip install -q ultralytics onnx onnxruntime onnxruntime-tools pandas matplotlib pyyaml

import os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/PyroPredict'
DRIVE_EXPORTS = f'{DRIVE_PROJECT}/exports'
DRIVE_METRICS = f'{DRIVE_PROJECT}/metrics'
os.makedirs(DRIVE_EXPORTS, exist_ok=True)
os.makedirs(DRIVE_METRICS, exist_ok=True)

print('Drive project:', DRIVE_PROJECT)

## 1 — Inputs and paths

In [ ]:
# Best model from Phase 4
FP32_ONNX = Path(f'{DRIVE_EXPORTS}/ablation_4_combined.onnx')

# Output INT8 model
INT8_ONNX = Path(f'{DRIVE_EXPORTS}/ablation_4_combined_int8.onnx')

# Dataset config generated in earlier phases
DATASET_YAML = Path(f'{DRIVE_PROJECT}/dataset.yaml')

if not FP32_ONNX.exists():
    raise FileNotFoundError(f'Missing FP32 model: {FP32_ONNX}')
if not DATASET_YAML.exists():
    raise FileNotFoundError(f'Missing dataset.yaml: {DATASET_YAML}')

print('FP32 model:', FP32_ONNX)
print('INT8 output:', INT8_ONNX)
print('Dataset YAML:', DATASET_YAML)

## 2 — Quantize FP32 ONNX to INT8 (dynamic quantization)

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

print('Quantizing...')
quantize_dynamic(
    model_input=str(FP32_ONNX),
    model_output=str(INT8_ONNX),
    weight_type=QuantType.QInt8,
)
print('Done:', INT8_ONNX)


## 3 — Model size comparison

In [ ]:
fp32_size_mb = FP32_ONNX.stat().st_size / (1024 * 1024)
int8_size_mb = INT8_ONNX.stat().st_size / (1024 * 1024)
size_reduction = (fp32_size_mb - int8_size_mb) / fp32_size_mb * 100

print(f'FP32 size: {fp32_size_mb:.2f} MB')
print(f'INT8 size: {int8_size_mb:.2f} MB')
print(f'Reduction: {size_reduction:.2f}%')


## 4 — Latency and throughput benchmark (ONNX Runtime CPU)

In [ ]:
import onnxruntime as ort
import cv2, yaml, random

with open(DATASET_YAML, 'r') as f:
    ds = yaml.safe_load(f)

base = Path(ds['path'])
test_rel = ds.get('test') or ds.get('val')
test_dir = base / test_rel

imgs = sorted([p for p in test_dir.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}])
if len(imgs) == 0:
    raise RuntimeError(f'No test images found in {test_dir}')

# Use up to 200 images for repeatable benchmark speed
random.seed(42)
sample_imgs = random.sample(imgs, min(200, len(imgs)))

def load_and_preprocess(img_path, input_hw):
    img = cv2.imread(str(img_path))
    h, w = input_hw
    img = cv2.resize(img, (w, h))
    x = img.astype(np.float32) / 255.0
    x = np.transpose(x, (2, 0, 1))[None, ...]
    return x

def benchmark_onnx(model_path, images):
    sess = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0]
    in_name = inp.name
    h = int(inp.shape[2]) if isinstance(inp.shape[2], int) else 640
    w = int(inp.shape[3]) if isinstance(inp.shape[3], int) else 640

    # warmup
    for p in images[:5]:
        x = load_and_preprocess(p, (h, w))
        _ = sess.run(None, {in_name: x})

    times = []
    for p in images:
        x = load_and_preprocess(p, (h, w))
        t0 = time.perf_counter()
        _ = sess.run(None, {in_name: x})
        times.append((time.perf_counter() - t0) * 1000)

    mean_ms = float(np.mean(times))
    std_ms = float(np.std(times))
    fps = 1000.0 / mean_ms
    return mean_ms, std_ms, fps

fp32_ms, fp32_std, fp32_fps = benchmark_onnx(FP32_ONNX, sample_imgs)
int8_ms, int8_std, int8_fps = benchmark_onnx(INT8_ONNX, sample_imgs)

print(f'FP32 latency: {fp32_ms:.2f} ± {fp32_std:.2f} ms, FPS: {fp32_fps:.2f}')
print(f'INT8 latency: {int8_ms:.2f} ± {int8_std:.2f} ms, FPS: {int8_fps:.2f}')
print(f'Speedup: {fp32_ms / int8_ms:.2f}x')


## 5 — Accuracy comparison (test set) via Ultralytics

In [ ]:
from ultralytics import YOLO

print('Evaluating FP32 ONNX on test split...')
fp32_model = YOLO(str(FP32_ONNX))
fp32_metrics = fp32_model.val(data=str(DATASET_YAML), split='test', imgsz=640, batch=16, plots=False)

print('Evaluating INT8 ONNX on test split...')
int8_model = YOLO(str(INT8_ONNX))
int8_metrics = int8_model.val(data=str(DATASET_YAML), split='test', imgsz=640, batch=16, plots=False)

print('Done accuracy evaluation')


## 6 — Build final comparison table

In [ ]:
comparison = pd.DataFrame([
    {
        'Model': 'FP32 ONNX',
        'Size (MB)': fp32_size_mb,
        'Latency (ms)': fp32_ms,
        'FPS': fp32_fps,
        'mAP@50': float(fp32_metrics.box.map50),
        'mAP@50:95': float(fp32_metrics.box.map),
        'Precision': float(fp32_metrics.box.mp),
        'Recall': float(fp32_metrics.box.mr),
    },
    {
        'Model': 'INT8 ONNX',
        'Size (MB)': int8_size_mb,
        'Latency (ms)': int8_ms,
        'FPS': int8_fps,
        'mAP@50': float(int8_metrics.box.map50),
        'mAP@50:95': float(int8_metrics.box.map),
        'Precision': float(int8_metrics.box.mp),
        'Recall': float(int8_metrics.box.mr),
    },
])

print('\n' + '═'*70)
print('PHASE 5 RESULTS — FP32 vs INT8')
print('═'*70)
print(comparison.to_string(index=False, float_format='{:.4f}'.format))
print('═'*70)

# Deltas
acc_drop = comparison.loc[0, 'mAP@50'] - comparison.loc[1, 'mAP@50']
lat_gain = comparison.loc[0, 'Latency (ms)'] / comparison.loc[1, 'Latency (ms)']
size_gain = (comparison.loc[0, 'Size (MB)'] - comparison.loc[1, 'Size (MB)']) / comparison.loc[0, 'Size (MB)'] * 100

print(f'\nAccuracy drop (mAP@50): {acc_drop:+.4f}')
print(f'Latency speedup: {lat_gain:.2f}x')
print(f'Size reduction: {size_gain:.2f}%')


## 7 — Save Phase 5 outputs to Drive

In [ ]:
comparison_path = Path(f'{DRIVE_METRICS}/quantization_comparison.csv')
summary_path = Path(f'{DRIVE_METRICS}/quantization_summary.json')
plot_path = Path(f'{DRIVE_METRICS}/quantization_comparison.png')

comparison.to_csv(comparison_path, index=False)

summary = {
    'fp32_size_mb': fp32_size_mb,
    'int8_size_mb': int8_size_mb,
    'size_reduction_pct': float(size_reduction),
    'fp32_latency_ms': fp32_ms,
    'int8_latency_ms': int8_ms,
    'latency_speedup_x': float(fp32_ms / int8_ms),
    'fp32_map50': float(comparison.loc[0, 'mAP@50']),
    'int8_map50': float(comparison.loc[1, 'mAP@50']),
    'map50_drop': float(comparison.loc[0, 'mAP@50'] - comparison.loc[1, 'mAP@50']),
}
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(['FP32', 'INT8'], [fp32_size_mb, int8_size_mb], color=['#3b82f6', '#10b981'])
axes[0].set_title('Model Size (MB)')

axes[1].bar(['FP32', 'INT8'], [fp32_ms, int8_ms], color=['#3b82f6', '#10b981'])
axes[1].set_title('Latency (ms/image)')

axes[2].bar(['FP32', 'INT8'], [comparison.loc[0, 'mAP@50'], comparison.loc[1, 'mAP@50']], color=['#3b82f6', '#10b981'])
axes[2].set_title('mAP@50')

plt.tight_layout()
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print('Saved:')
print(' -', comparison_path)
print(' -', summary_path)
print(' -', plot_path)


In [ ]:
print('\n' + '═'*70)
print('PHASE 5 COMPLETE')
print('═'*70)
print('Next: Phase 6 Streamlit app + Docker demo packaging')
